In [1]:
import duckdb
import requests
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

In [2]:
conn = duckdb.connect("v2_database.duckdb")
df_gen = conn.execute("select * from generations").fetch_df()
conn.close()

# Filter the first row where 'status' is blank (NaN or empty string)
run_gen = df_gen[df_gen['STATUS'].isna() | (df_gen['STATUS'] == '')].iloc[0]['NAME']

# Show value 'name'
print(f'Running for Gen: {run_gen}')

Running for Gen: generation-i


In [3]:
# URL da API para a geração específica
url = f"https://pokeapi.co/api/v2/generation/{run_gen}/"

# Requisição para obter os dados da geração
response = requests.get(url)

# Verificar se a requisição foi bem-sucedida
if response.status_code == 200:
    data = response.json()
    
    # Extrair a lista de Pokémon da geração
    pokemon_species = data.get("pokemon_species", [])
    
    # Extrair o nome e o ID do Pokémon a partir da URL
    pokemon_data = [
        {
            "id": int(pokemon["url"].rstrip("/").split("/")[-1])  # Extrair o ID da URL
        }
        for pokemon in pokemon_species
    ]
    
    # Criar o DataFrame
    df_ids = pd.DataFrame(pokemon_data)
    df_ids = df_ids.sort_values(by='id', ascending=True)

    # Obter o número de linhas e colunas do DataFrame
    n_rows, n_cols = df_ids.shape

    # Exibir o resultado
    print(f"Dataframe has {n_rows} rows and {n_cols} columns.")
    
else:
    print(f"Erro ao acessar a API: {response.status_code}")

Dataframe has 151 rows and 1 columns.


In [4]:
# URL base da PokeAPI
api_url = "https://pokeapi.co/api/v2/pokemon/"

# Função para buscar os stats de um único Pokémon
def fetch_pokemon_stats(pokemon_id):
    try:
        response = requests.get(f"{api_url}{pokemon_id}", timeout=5)
        response.raise_for_status()
        data = response.json()
        
        stats_data = []
        stats = data.get("stats", [])
        
        # Iterar pelas estatísticas, desconsiderando a coluna 'effort'
        for stat_info in stats:
            stats_data.append({
                "id": pokemon_id,
                "stat": stat_info["stat"]["name"],
                "base_stat": stat_info["base_stat"]  # Apenas 'base_stat'
            })
        return stats_data
    
    except requests.RequestException as e:
        print(f"Erro ao acessar o Pokémon ID {pokemon_id}: {e}")
        return []  # Retorna uma lista vazia em caso de erro

# Função principal para buscar os stats usando múltiplas threads
def fetch_pokemon_stats_expanded(df_ids, max_workers=10):
    stats_data = []
    
    # Usar ThreadPoolExecutor para paralelizar as requisições
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submete as tarefas para execução paralela
        futures = {executor.submit(fetch_pokemon_stats, pokemon_id): pokemon_id for pokemon_id in df_ids['id']}
        
        # Processa as respostas à medida que forem concluídas
        for future in as_completed(futures):
            result = future.result()
            if result:  # Verifica se há dados retornados
                stats_data.extend(result)
    
    # Retorna um DataFrame com os dados coletados
    return pd.DataFrame(stats_data)

# Coletar os dados de stats com base no DataFrame de IDs
df_stats_expanded = fetch_pokemon_stats_expanded(df_ids, max_workers=10)

In [5]:
# Create DuckDB database
conn = duckdb.connect("v2_database.duckdb")
conn.execute("""
    CREATE TABLE IF NOT EXISTS b_stats ( 
      id INT,
      stat TEXT,
      base_stat INT
    )
""")
conn.execute("""INSERT INTO b_stats SELECT * FROM df_stats_expanded""")
conn.close()